In [1]:
!nvidia-smi

Tue Oct 13 16:41:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### **1. Install Dependencies**

In [2]:
!pip install -q vllm
!pip install -q pyngrok
!pip install -q torch transformers
!pip install -U huggingface_hub
!pip uninstall -y torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.

In [4]:
# Keep credentials out of the notebook. In Colab, add NGROK_AUTH_TOKEN under Secrets (key icon).
from google.colab import userdata
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
if not ngrok_token:
    raise ValueError("Add your Ngrok token to Colab Secrets as NGROK_AUTH_TOKEN, then rerun this cell.")
ngrok.set_auth_token(ngrok_token)
print("Ngrok authenticated.")


Ngrok authenticated.


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cu130
CUDA: 13.0
GPU: Tesla T4


In [ ]:
# !pip uninstall -y torchaudio

In [6]:
import vllm

print("vLLM:", vllm.__version__)

vLLM: 0.31.0


### **2. Download the model**

Model: `Qwen/Qwen3-4B-Instruct-2507`  
Serving: vLLM OpenAI-compatible API  
Quantisation: BitsAndBytes 4-bit (NF4 where supported by the installed vLLM/BitsAndBytes stack)  
Generation defaults: temperature `0.0`, seed `42`, maximum new tokens `600`.


In [ ]:
# Optional legacy model reference (not used by the active setup)
# !hf download meta-llama/Llama-2-7b-hf --local-dir /content/models/Llama-2-7b-hf


In [7]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_DIR = "/content/models/Qwen3-4B-Instruct-2507"
TEMPERATURE = 0.0
SEED = 42
MAX_NEW_TOKENS = 600

!hf download Qwen/Qwen3-4B-Instruct-2507 --local-dir /content/models/Qwen3-4B-Instruct-2507


Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.57k [00:00<?, ?B/s]         

Fetching 13 files:   8% 1/13 [00:00<00:01,  7.72it/s]
Reconstructing (incomplete total...):  87% 1.57k/1.81k [00:00<00:00, 12.7kB/s]
Reconstructing (incomplete total...):  71% 1.81k/2.54k [00:00<00:00, 12.7kB/s]

Fetching 13 files:  23% 3/13 [00:00<00:00, 13.42it/s]
Reconstructing (incomplete total...):   0% 2.54k/1.67M [00:00<02:11, 12.7kB/s]
Reconstructing (incomplete total...):  98% 1.67M/1.71M [00:00<00:00, 6.06MB/s]
Reconstructing (incomplete total...): 100% 1.71M/1.72M [00:00<00:00, 6.06MB/s]
Reconstructing (incomplete total...):

In [8]:
!ls -lh /content/models/Qwen3-4B-Instruct-2507


total 7.6G
-rw-r--r-- 1 root root  727 Oct 13 16:48 config.json
-rw-r--r-- 1 root root  238 Oct 13 16:48 generation_config.json
-rw-r--r-- 1 root root  12K Oct 13 16:48 LICENSE
-rw-r--r-- 1 root root 1.6M Oct 13 16:48 merges.txt
-rw-r--r-- 1 root root 3.7G Oct 13 16:49 model-00001-of-00003.safetensors
-rw-r--r-- 1 root root 3.8G Oct 13 16:49 model-00002-of-00003.safetensors
-rw-r--r-- 1 root root  96M Oct 13 16:48 model-00003-of-00003.safetensors
-rw-r--r-- 1 root root  33K Oct 13 16:48 model.safetensors.index.json
-rw-r--r-- 1 root root 8.0K Oct 13 16:48 README.md
-rw-r--r-- 1 root root 9.2K Oct 13 16:48 tokenizer_config.json
-rw-r--r-- 1 root root  11M Oct 13 16:48 tokenizer.json
-rw-r--r-- 1 root root 2.7M Oct 13 16:48 vocab.json


### **4. Start vLLM using Qwen3-4B with 4-bit quantisation**

This uses vLLM's BitsAndBytes backend rather than Transformers' `BitsAndBytesConfig`. The installed vLLM version must support BitsAndBytes quantisation. If the backend/quantisation option is unavailable, upgrade vLLM to a compatible release and restart the Colab runtime.


In [10]:
%pip install "huggingface-hub>=1.5.0,<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.2.0
    Uninstalling huggingface_hub-2.2.0:
      Successfully uninstalled huggingface_hub-2.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.31.0 requires torchaudio==2.11.0, which is not installed.


In [11]:
# !python -m vllm.entrypoints.openai.api_server \
#     --model /content/models/Qwen3-4B-Instruct-2507 \
#     --served-model-name Qwen/Qwen3-4B-Instruct-2507 \
#     --quantization bitsandbytes \
#     --host 0.0.0.0 \
#     --port 8000 \
#     --gpu-memory-utilization 0.85


/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/openai/api_server.py:24: DeprecationWarning: `vllm.entrypoints.openai.api_server` is deprecated and will likely beunsupported in a future version. Use the corresponding function from `vllm.entrypoints.launchers` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/openai/api_server.py:51: DeprecationWarning: The `python -m vllm.entrypoints.openai.api_server` command is deprecated and may be removed in a future release. Please use `vllm server` instead.
  warnings.warn(
(APIServer pid=3711) INFO 10-13 16:51:35 [api_utils.py:347] 
(APIServer pid=3711) INFO 10-13 16:51:35 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=3711) INFO 10-13 16:51:35 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=3711) INFO 10-13 16:51:35 [api_utils.py:347]   █▄█▀ █     █     █     █  model   /content/models/Qwen3-4B-Instruct-2507
(APIServer pid=3711) INFO 10-13 16:51:35 [api_utils.p

In [ ]:
!vllm serve /content/models/Qwen3-4B-Instruct-2507 \
    --host 0.0.0.0 \
    --port 8000 \
    --served-model-name Qwen/Qwen3-4B-Instruct-2507 \
    --dtype half \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --generation-config vllm

(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347] 
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347]   █▄█▀ █     █     █     █  model   /content/models/Qwen3-4B-Instruct-2507
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:347] 
(APIServer pid=4269) INFO 10-13 16:53:33 [api_utils.py:286] non-default args: {'model_tag': '/content/models/Qwen3-4B-Instruct-2507', 'host': '0.0.0.0', 'model': '/content/models/Qwen3-4B-Instruct-2507', 'dtype': 'half', 'max_model_len': 8192, 'served_model_name': ['Qwen/Qwen3-4B-Instruct-2507'], 'generation_config': 'vllm', 'gpu_memory_utilization': 0.85}
(APIServer pid=4269) INFO 10-13 16:53:33 [model.py:699] Resolved architecture: Qwen3ForCausalLM
(APIServer 

In [ ]:
# Expose with Ngrok

!ngrok http 8000     # -----> bash

In [ ]:
!curl http://localhost:8000/v1/models

### **5. Test vLLM directly in Colab**

In [ ]:
import requests

response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "What is an invoice?"}],
        "max_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "seed": SEED
    },
    timeout=180
)
print(response.status_code)
response.raise_for_status()
print(response.json())


### **6. Now create/recreate ngrok**

In [ ]:
from pyngrok import ngrok

ngrok.kill()
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print("PUBLIC URL:", PUBLIC_URL)
print("API base:", PUBLIC_URL + "/v1")


### **7. Test ngrok from Colab**

In [ ]:
import requests

response = requests.get(PUBLIC_URL + "/v1/models", timeout=30)
print(response.status_code)
response.raise_for_status()
print(response.json())


In [ ]:
import requests

remote_response = requests.post(
    PUBLIC_URL + "/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "What is an invoice?"}],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "max_tokens": MAX_NEW_TOKENS
    },
    timeout=180
)
print(remote_response.status_code)
remote_response.raise_for_status()
print(remote_response.json())


In [ ]:
# Optional background-server alternative. Run only if you have stopped the foreground server in cell 12.
# import subprocess, time
# process = subprocess.Popen([
#     "python", "-m", "vllm.entrypoints.openai.api_server",
#     "--model", MODEL_DIR,
#     "--served-model-name", MODEL_ID,
#     "--quantization", "bitsandbytes",
#     "--host", "0.0.0.0", "--port", "8000",
#     "--gpu-memory-utilization", "0.85"
# ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# time.sleep(30)
# print("Server process status:", process.poll())


In [ ]:
# Background server option (use this OR the foreground server in cell 12, not both).
import subprocess
import time

process = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_DIR,
        "--served-model-name", MODEL_ID,
        "--quantization", "bitsandbytes",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--gpu-memory-utilization", "0.85"
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(30)
print("Checking vLLM...")
print("Process status (None means still running):", process.poll())
if process.poll() is not None:
    print(process.stdout.read()[-6000:])


In [ ]:
from pyngrok import ngrok
import time

# Create the tunnel after the vLLM server is running.
public_url = ngrok.connect(8000).public_url
PUBLIC_URL = public_url
print("PUBLIC ENDPOINT:", PUBLIC_URL)
print("OpenAI-compatible API:", PUBLIC_URL + "/v1")
print(f'Use BASE_URL = "{PUBLIC_URL}/v1" in your client.')


In [ ]:
# One-shot health check (instead of an infinite loop that prevents later cells from running).
import requests
from datetime import datetime

health_url = PUBLIC_URL + "/v1/models"
try:
    r = requests.get(health_url, timeout=15)
    print(datetime.now().strftime("%H:%M:%S"), "HTTP", r.status_code)
    r.raise_for_status()
    print("vLLM endpoint is responding.")
except requests.RequestException as exc:
    print("Endpoint health check failed:", exc)


### **Test Ngrok tunnel**

In [ ]:
!pip install -q vllm pyngrok


In [ ]:
# Authenticate securely using Colab Secrets; never paste the token into a notebook cell.
from google.colab import userdata
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
if not ngrok_token:
    raise ValueError("Add your Ngrok token to Colab Secrets as NGROK_AUTH_TOKEN.")
ngrok.set_auth_token(ngrok_token)


In [ ]:
# Legacy duplicate server cell updated to the selected model. Run only if no server is already running.
import subprocess, time
process = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_DIR,
    "--served-model-name", MODEL_ID,
    "--quantization", "bitsandbytes",
    "--host", "0.0.0.0",
    "--gpu-memory-utilization", "0.85",
    "--port", "8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(30)
print("Server process status (None means running):", process.poll())
if process.poll() is not None:
    print(process.stdout.read()[-6000:])


In [ ]:
from pyngrok import ngrok
url = ngrok.connect(8000).public_url
PUBLIC_URL = url
print("PUBLIC URL:", url)
print("API base:", url + "/v1")
